# Ecological embeddings of the human gut — quickstart

Loads the SNE vectors, looks at a taxon's ecological neighbours, and scores a
sample with the same model the website runs. The last cell checks that the
score you get here matches the one the site reports, so you can confirm the
two agree before building on either.

Set `BASE` below to wherever the files are served from.

In [ ]:
BASE = "https://microbiome.example.org/data"

%pip install --quiet gensim onnxruntime numpy

## The vectors

14,093 OTUs, 100 dimensions, word2vec text. Cosine similarity is the metric
throughout: the embedding was trained so that direction carries the meaning.

In [ ]:
import urllib.request
from gensim.models import KeyedVectors

urllib.request.urlretrieve(f"{BASE}/download/sne_vectors.txt.gz", "sne_vectors.txt.gz")
vectors = KeyedVectors.load_word2vec_format("sne_vectors.txt.gz", binary=False)
print(len(vectors.index_to_key), "OTUs x", vectors.vector_size, "dimensions")

In [ ]:
# Ecological neighbours of one OTU: who it co-occurs with, not who it is
# related to. The two lists overlap far less than a phylogeny would suggest,
# which is the paper's point.
probe = "AB002518.1.1416"
for otu, similarity in vectors.most_similar(probe, topn=10):
    print(f"{similarity:.3f}  {otu}")

## Scoring a sample

The classifier is a 13-fold leave-one-disease-out ensemble. Its input is not
token indices but already-gathered 100-dimensional vectors: all thirteen
folds share one frozen embedding table, so the graph takes the gathered rows
and the table ships as a separate file. That is exactly what the browser
does.

Preprocessing, in three steps: rank the sample's non-zero counts, divide by
the largest rank, keep the 600 most abundant OTUs. Getting this wrong is the
easy mistake, so it is spelled out here rather than imported.

In [ ]:
import json
import numpy as np
import onnxruntime as ort

vocab = json.loads(urllib.request.urlopen(f"{BASE}/vocab.json").read())
index = {otu: i for i, otu in enumerate(vocab["ids"])}
num_steps, d_model = vocab["num_steps"], vocab["d_model"]

urllib.request.urlretrieve(f"{BASE}/dysbiosis_embed.f16.bin", "embed.f16.bin")
urllib.request.urlretrieve(f"{BASE}/dysbiosis_encoder.onnx", "encoder.onnx")
embedding = np.fromfile("embed.f16.bin", dtype=np.float16).astype(np.float32)
embedding = embedding.reshape(vocab["n_tokens"], d_model)

session = ort.InferenceSession("encoder.onnx", providers=["CPUExecutionProvider"])
print("vocabulary", len(vocab["ids"]), "OTUs; sequence length", num_steps)

In [ ]:
def preprocess(counts):
    """counts: {otu_id: count} for one sample."""
    ids = list(counts)
    values = np.array([counts[otu] for otu in ids], dtype=np.float64)

    # Only non-zero entries are ranked, ties share their average rank, and the
    # whole sample is divided by its largest rank. Zeros stay zero.
    nonzero = np.nonzero(values)[0]
    present = values[nonzero]
    order = np.argsort(present, kind="stable")
    ranks = np.empty(present.size)
    start = 0
    while start < order.size:
        stop = start + 1
        while stop < order.size and present[order[stop]] == present[order[start]]:
            stop += 1
        ranks[order[start:stop]] = (start + stop - 1) / 2 + 1
        start = stop

    abundance = np.zeros_like(values)
    abundance[nonzero] = ranks / ranks.max()

    # The 600 most abundant, or every non-zero OTU when there are fewer.
    # Taking 600 regardless would fill the tail with abundance-zero positions
    # that are not masked, changing the pooling denominator.
    if nonzero.size >= num_steps:
        take = np.argsort(-abundance, kind="stable")[:num_steps]
    else:
        take = nonzero

    features = np.zeros(num_steps, dtype=np.int64)
    weight = np.zeros(num_steps, dtype=np.float32)
    mask = np.zeros(num_steps, dtype=np.int64)
    for position, k in enumerate(take):
        features[position] = index.get(ids[k], -1) + 2      # not found -> <unk>
        weight[position] = abundance[k]
        mask[position] = 0 if features[position] in (0, 1) else 1
    return features, weight, mask

In [ ]:
def score(counts):
    features, weight, mask = preprocess(counts)
    gathered = embedding[features][None, :, :]
    logit, _ = session.run(None, {
        "inputs": gathered,
        "weight": weight[None, :],
        "mask": mask.astype(np.int64)[None, :],
    })
    return float(logit[0][0])


example = json.loads(urllib.request.urlopen(
    f"{BASE}/examples/ibd_case.json").read())
logit = score(example["counts"])
print(f"{example['label']} ({example['sample_id']}): logit {logit:.4f}")
print(f"the site reports {example['expected_logit']:.4f}")
assert abs(logit - example["expected_logit"]) < 1e-3, (
    "the notebook and the site disagree, so the preprocessing here has "
    "drifted from the site's")
print("agrees with the site")

In [ ]:
# Turn the logit into the percentile the site shows.
ref = json.loads(urllib.request.urlopen(f"{BASE}/ref_scores.json").read())
all_scores = np.sort(np.array(ref["controls"] + ref["cases"]))
percentile = np.searchsorted(all_scores, logit) / len(all_scores) * 100
print(f"{percentile:.0f}th percentile of {len(all_scores)} reference samples")
print(f"{ref['n_controls']} controls, {ref['n_cases']} cases")

## What the number means

The reference cohort is scored by the same ensemble, so for a sample from a
disease *inside* the cohort the percentile is well calibrated. The model's
accuracy on a disease it has never seen is lower — AUC 0.64 by
leave-one-disease-out against 0.80 within the cohort — so a sample from a
condition outside the thirteen will sit closer to the middle than it should.

This is a research tool. It is not a medical device and it does not diagnose
anything.